# 01 – Download & EDA
Download SmartDoc-QA, verify SD7K structure, run exploratory analysis.

In [ ]:
# ─── Install dependencies ───────────────────────────────────────────────────
!pip -q install segmentation-models-pytorch albumentations jiwer pytesseract \
  scikit-image pytorch-msssim gradio easyocr pyyaml tqdm matplotlib seaborn joblib
# On Colab: also install tesseract binary
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', 'tesseract-ocr'], check=True)

In [ ]:
# ─── Mount Google Drive (Colab) ─────────────────────────────────────────────
import sys
COLAB = 'google.colab' in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRescue'
else:
    PROJECT_ROOT = '.'  # local: repo root
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ─── Download SmartDoc-QA from Zenodo ───────────────────────────────────────
import os

SMARTDOC_URL = 'https://zenodo.org/records/5293201/files/Dataset%20SmartDoc-QA.zip?download=1'
SMARTDOC_DIR = os.path.join(PROJECT_ROOT, 'data', 'smartdoc_qa')
zip_path = os.path.join(PROJECT_ROOT, 'data', 'Dataset_SmartDoc-QA.zip')

# Automatically clean up any previous corrupted/incomplete download
bad_old = os.path.join(PROJECT_ROOT, 'data', 'SmartDocQA.zip')
if os.path.exists(bad_old):
    print(f'Removing previous corrupted file: {bad_old}')
    os.remove(bad_old)

if os.path.exists(zip_path) and os.path.getsize(zip_path) < 1_000_000_000:
    print('Removing incomplete download …')
    os.remove(zip_path)

os.makedirs(SMARTDOC_DIR, exist_ok=True)

if not os.path.exists(zip_path):
    print('Downloading SmartDoc-QA (~13.6 GB) to Google Drive …')
    !wget -c --show-progress '{SMARTDOC_URL}' -O '{zip_path}'
else:
    print('SmartDoc-QA zip archive already downloaded.')
    
print('Extracting to Google Drive … (this takes a few minutes for 13.6 GB)')
!unzip -q -o '{zip_path}' -d '{SMARTDOC_DIR}'
print('Done! SmartDoc-QA extracted successfully.')

In [ ]:
# ─── Verify SD7K structure ────────────────────────────────────────────────────
# Works for .png, .jpg, or any image extension
from pathlib import Path

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def count_images(directory):
    p = Path(directory)
    if not p.exists():
        return 0
    return sum(1 for f in p.iterdir() if f.suffix.lower() in IMG_EXTS)

SD7K_RAW = os.path.join(PROJECT_ROOT, 'data', 'sd7k_raw')

for split in ['train', 'test']:
    n_i = count_images(os.path.join(SD7K_RAW, split, 'input'))
    n_t = count_images(os.path.join(SD7K_RAW, split, 'target'))
    print(f'{split}: {n_i} input, {n_t} target')

In [ ]:
# ─── EDA: Visualise SD7K pairs ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import random
from PIL import Image

train_input_dir = Path(SD7K_RAW) / 'train' / 'input'
train_target_dir = Path(SD7K_RAW) / 'train' / 'target'

# Collect all image files regardless of extension
input_files = sorted(p.name for p in train_input_dir.iterdir() if p.suffix.lower() in IMG_EXTS) if train_input_dir.exists() else []
if input_files:
    samples = random.sample(input_files, min(6, len(input_files)))

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    for col, fn in enumerate(samples):
        s = Image.open(train_input_dir / fn).resize((256, 256))
        c = Image.open(train_target_dir / fn).resize((256, 256))
        axes[0, col].imshow(s); axes[0, col].set_title(fn[:15], fontsize=7); axes[0, col].axis('off')
        axes[1, col].imshow(c); axes[1, col].axis('off')
    axes[0, 0].set_ylabel('Input (shadow)', fontsize=9)
    axes[1, 0].set_ylabel('Target (shadow-free)', fontsize=9)
    plt.suptitle('SD7K Training Pairs (random sample)', fontsize=12)
    plt.tight_layout()
    os.makedirs(os.path.join(PROJECT_ROOT, 'results', 'figures'), exist_ok=True)
    plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'sd7k_sample_pairs.png'), dpi=100)
    plt.show()
else:
    print(f'No SD7K training images found at {train_input_dir}. Skipping plot.')

In [ ]:
# ─── EDA: Image size distribution ───────────────────────────────────────────
import numpy as np
import seaborn as sns

if input_files:
    widths, heights = [], []
    for fn in input_files[:500]:  # sample up to 500
        with Image.open(train_input_dir / fn) as img:
            w, h = img.size
            widths.append(w); heights.append(h)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(widths, ax=axes[0], bins=20, color='steelblue')
    axes[0].set_title('Width distribution'); axes[0].set_xlabel('pixels')
    sns.histplot(heights, ax=axes[1], bins=20, color='coral')
    axes[1].set_title('Height distribution'); axes[1].set_xlabel('pixels')
    plt.tight_layout(); plt.show()
    print(f'Median size: {int(np.median(widths))}×{int(np.median(heights))}')
else:
    print('No images available for size distribution analysis.')

In [ ]:
# ─── EDA: Pixel statistics (brightness before/after) ────────────────────────
if input_files:
    means_input, means_target = [], []
    for fn in random.sample(input_files, min(200, len(input_files))):
        s = np.array(Image.open(train_input_dir / fn).convert('L'))
        c = np.array(Image.open(train_target_dir / fn).convert('L'))
        means_input.append(s.mean())
        means_target.append(c.mean())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(means_input,  bins=30, alpha=0.6, label='Input (shadow)', color='navy')
    ax.hist(means_target, bins=30, alpha=0.6, label='Target (shadow-free)', color='orange')
    ax.set_xlabel('Mean pixel intensity'); ax.set_ylabel('Count')
    ax.set_title('Brightness distribution: input vs target')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('No images available for pixel statistics.')